In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA 0 — Setup de entorno y extracción del dataset               ║
# ║  Detecta PROJECT_ROOT automáticamente y descomprime isic_2019.7z   ║
# ║  Coloca isic_2019.7z en: <PROJECT_ROOT>/datasets/isic_2019/        ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os
import subprocess
from pathlib import Path

# ── Detección automática de PROJECT_ROOT ────────────────────────────
# El notebook vive en PROJECT_ROOT/notebooks/. Jupyter suele lanzar
# con cwd = directorio del notebook, pero puede variar según entorno.
_cwd = Path(os.getcwd()).resolve()
if _cwd.name == "notebooks":
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "notebooks").exists() and (_cwd / "src").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "notebooks").exists() and (_cwd.parent / "src").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd  # fallback: directorio actual
print(f"PROJECT_ROOT → {PROJECT_ROOT}")

# ── Crear carpeta del dataset ────────────────────────────────────────
ISIC_DIR = PROJECT_ROOT / "datasets" / "isic_2019"
ISIC_DIR.mkdir(parents=True, exist_ok=True)

# ── Verificar si ya está extraído ───────────────────────────────────
_marker = ISIC_DIR / "ISIC_2019_Training_Input_preprocessed"
if _marker.exists():
    print("✓ Dataset ISIC ya extraído — saltando descompresión")
else:
    _archive = ISIC_DIR / "isic_2019.7z"
    if not _archive.exists():
        raise FileNotFoundError(
            f"\n{'━'*60}\n"
            f"  Archivo no encontrado: {_archive}\n"
            f"  Coloca isic_2019.7z en:\n"
            f"    {ISIC_DIR}\n"
            f"{'━'*60}"
        )
    _size_mb = _archive.stat().st_size / 1e6
    print(f"Extrayendo {_archive.name} ({_size_mb:.0f} MB) → {ISIC_DIR} ...")
    _result = subprocess.run(
        ["7z", "x", str(_archive), f"-o{ISIC_DIR}", "-y"],
        capture_output=True, text=True,
    )
    if _result.returncode != 0:
        raise RuntimeError(
            f"Error al extraer {_archive.name}:\n{_result.stderr}"
        )
    print("✓ Extracción completada")

# ── Verificar estructura extraída ────────────────────────────────────
_required = {
    "Imágenes preprocesadas": ISIC_DIR / "ISIC_2019_Training_Input_preprocessed",
    "Train CSV":              ISIC_DIR / "splits" / "isic_train.csv",
    "Val CSV":                ISIC_DIR / "splits" / "isic_val.csv",
    "Test CSV":               ISIC_DIR / "splits" / "isic_test.csv",
    "GroundTruth CSV":        ISIC_DIR / "ISIC_2019_Training_GroundTruth.csv",
}
_missing = []
for _desc, _path in _required.items():
    _ok = _path.exists()
    print(f"  {'✓' if _ok else '✗'} {_desc}: {_path.relative_to(PROJECT_ROOT)}")
    if not _ok:
        _missing.append(_desc)
if _missing:
    raise FileNotFoundError(
        f"Faltan archivos tras la extracción: {_missing}\n"
        f"Verifica que isic_2019.7z esté íntegro."
    )
print("\n✓ Dataset ISIC 2019 listo")

# Expert 2 — ConvNeXt-Small / ISIC 2019

**Notebook 100% autosuficiente** para entrenar Expert 2 del pipeline de clasificación médica multi-experto.

- **Backbone:** ConvNeXt-Small (timm, pesos ImageNet)
- **Dataset:** ISIC 2019 — 8 clases de lesiones dermatoscópicas
- **Entrenamiento trifásico:**
  - Fase 1 (épocas 1–5): Solo head, backbone congelado
  - Fase 2 (épocas 6–20): Fine-tuning diferencial (últimos 2 stages + head)
  - Fase 3 (épocas 21–40): Full fine-tuning + early stopping
- **Loss:** FocalLossMultiClass con pesos de clase y label smoothing
- **Augmentación:** RandomCrop, flips, rotación 360°, ColorJitter, RandomGamma, ShadesOfGray, CoarseDropout, CutMix/MixUp

Clases ISIC 2019:
```
0=MEL  1=NV  2=BCC  3=AK  4=BKL  5=DF  6=VASC  7=SCC
```

---

**Instrucciones:** Configura `PROJECT_ROOT` en la celda de configuración y ejecuta todas las celdas de arriba a abajo.

## 0. Instalación de dependencias

In [ ]:
!pip install timm albumentations pandas scikit-learn tqdm

## 1. Imports

In [ ]:
import os
import json
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

import timm
from timm.layers import trunc_normal_

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("expert2_notebook")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Configuración de rutas y constantes de módulos del proyecto

In [ ]:
# PROJECT_ROOT fue detectado en la Celda 0 (Setup & Extracción).
# Si ejecutas esta celda de forma aislada, descomenta la siguiente línea:
# PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())

# Constantes que en el proyecto se importan de módulos internos.
# Aquí se definen inline para que el notebook sea autosuficiente.
EXPERT_IDS = {"isic": 2, "luna": 3}

# Rutas de datos — SOLO imágenes preprocesadas (DullRazor + resize shorter_side=224)
ISIC_DATA_DIR = PROJECT_ROOT / "datasets" / "isic_2019" / "ISIC_2019_Training_Input_preprocessed"
ISIC_TRAIN_CSV = PROJECT_ROOT / "datasets" / "isic_2019" / "splits" / "isic_train.csv"
ISIC_VAL_CSV = PROJECT_ROOT / "datasets" / "isic_2019" / "splits" / "isic_val.csv"
ISIC_TEST_CSV = PROJECT_ROOT / "datasets" / "isic_2019" / "splits" / "isic_test.csv"

# Checkpoint de salida (bug corregido: era expert_01_efficientnet_b3)
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "expert_01_convnext_small"
CHECKPOINT_PATH = CHECKPOINT_DIR / "expert2_best.pt"
TRAINING_LOG_PATH = CHECKPOINT_DIR / "expert2_training_log.json"

# Verificar rutas — las imágenes preprocesadas son OBLIGATORIAS
if not ISIC_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Directorio de imágenes preprocesadas no encontrado: {ISIC_DATA_DIR}\n"
        f"Descarga las imágenes preprocesadas (DullRazor + resize) desde Kaggle."
    )
print(f"✓ ISIC_DATA_DIR: {ISIC_DATA_DIR}")

for label, path in [
    ("ISIC_TRAIN_CSV", ISIC_TRAIN_CSV),
    ("ISIC_VAL_CSV", ISIC_VAL_CSV),
    ("ISIC_TEST_CSV", ISIC_TEST_CSV),
]:
    if not path.exists():
        print(f"⚠ {label} no encontrado: {path}")
    else:
        print(f"✓ {label}: {path}")

print(f"\nCheckpoint se guardará en: {CHECKPOINT_PATH}")

## 3. Hiperparámetros (inline desde expert2_config.py)

In [ ]:
# ── GENERAL ──────────────────────────────────────────────────────────
EXPERT2_NUM_CLASSES: int = 8
EXPERT2_IMG_SIZE: int = 224
EXPERT2_BATCH_SIZE: int = 32
EXPERT2_ACCUMULATION_STEPS: int = 3   # batch efectivo = 96
EXPERT2_NUM_WORKERS: int = 4
EXPERT2_LABEL_SMOOTHING: float = 0.1
EXPERT2_MONITOR: str = "val_f1_macro"  # métrica para checkpoint

# ── FASE 1 (épocas 1-5): solo head, backbone congelado ──────────────
EXPERT2_PHASE1_EPOCHS: int = 5
EXPERT2_PHASE1_LR: float = 3e-4
EXPERT2_PHASE1_WD: float = 1e-4
EXPERT2_PHASE1_ETA_MIN: float = 3e-5  # CosineAnnealingLR eta_min

# ── FASE 2 (épocas 6-20): fine-tuning diferencial ───────────────────
EXPERT2_PHASE2_EPOCHS: int = 15
EXPERT2_PHASE2_HEAD_LR: float = 3e-4
EXPERT2_PHASE2_BACKBONE_LR: float = 1e-5
EXPERT2_PHASE2_WD: float = 1e-4
EXPERT2_PHASE2_T0: int = 10   # CosineAnnealingWarmRestarts T_0
EXPERT2_PHASE2_T_MULT: int = 2
EXPERT2_PHASE2_ETA_MIN: float = 1e-7

# ── FASE 3 (épocas 21-40): full fine-tuning + early stopping ────────
EXPERT2_PHASE3_EPOCHS: int = 20
EXPERT2_PHASE3_HEAD_LR: float = 1e-4
EXPERT2_PHASE3_BACKBONE_LR: float = 5e-6
EXPERT2_PHASE3_WD: float = 1e-4
EXPERT2_PHASE3_T0: int = 10   # CosineAnnealingWarmRestarts T_0
EXPERT2_PHASE3_T_MULT: int = 2
EXPERT2_PHASE3_ETA_MIN: float = 1e-7
EXPERT2_EARLY_STOPPING_PATIENCE: int = 8

# ── Derivada ─────────────────────────────────────────────────────────
EXPERT2_TOTAL_EPOCHS: int = (
    EXPERT2_PHASE1_EPOCHS + EXPERT2_PHASE2_EPOCHS + EXPERT2_PHASE3_EPOCHS
)

# ── Constantes de entrenamiento ─────────────────────────────────────
SEED = 42
MIN_DELTA = 0.001
GRAD_CLIP_NORM = 1.0

print(f"Total épocas: {EXPERT2_TOTAL_EPOCHS}")
print(f"Batch efectivo: {EXPERT2_BATCH_SIZE} × {EXPERT2_ACCUMULATION_STEPS} = {EXPERT2_BATCH_SIZE * EXPERT2_ACCUMULATION_STEPS}")
print(f"Fases: head-only({EXPERT2_PHASE1_EPOCHS}) → partial({EXPERT2_PHASE2_EPOCHS}) → full({EXPERT2_PHASE3_EPOCHS})")

## 4. Custom Transforms (RandomGamma, ShadesOfGray, CoarseDropout)

In [ ]:
TARGET_SIZE = EXPERT2_IMG_SIZE  # 224


class RandomGamma:
    """Corrección gamma aleatoria: I' = I^γ con γ ∈ gamma_range."""

    def __init__(self, gamma_range: tuple[float, float] = (0.7, 1.5), p: float = 0.5):
        self.gamma_range = gamma_range
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.random() < self.p:
            gamma = np.random.uniform(*self.gamma_range)
            return transforms.functional.adjust_gamma(img, gamma)
        return img


class ShadesOfGray:
    """Color Constancy: normalización del iluminante por norma de Minkowski (p=6).
    Aplica corrección von Kries diagonal para simular luz blanca canónica."""

    def __init__(self, power: int = 6):
        self.power = power

    def __call__(self, img: Image.Image) -> Image.Image:
        arr = np.array(img, dtype=np.float32)
        illuminant = np.mean(arr**self.power, axis=(0, 1)) ** (1.0 / self.power)
        illuminant = illuminant / (np.mean(illuminant) + 1e-6)
        corrected = arr / (illuminant + 1e-6)
        return Image.fromarray(np.clip(corrected, 0, 255).astype(np.uint8))


class CoarseDropout:
    """Elimina 1–3 parches rectangulares aleatorios (CutOut / CoarseDropout).
    Opera sobre PIL Image ANTES de ToTensor."""

    def __init__(
        self,
        n_holes_range: tuple[int, int] = (1, 3),
        size_range: tuple[int, int] = (32, 96),
        fill_value: tuple[int, int, int] = (
            int(0.485 * 255),
            int(0.456 * 255),
            int(0.406 * 255),
        ),
        p: float = 0.5,
    ):
        self.n_holes_range = n_holes_range
        self.size_range = size_range
        self.fill_value = fill_value
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if np.random.random() >= self.p:
            return img
        arr = np.array(img)
        h, w = arr.shape[:2]
        n_holes = np.random.randint(self.n_holes_range[0], self.n_holes_range[1] + 1)
        for _ in range(n_holes):
            hole_h = np.random.randint(self.size_range[0], self.size_range[1] + 1)
            hole_w = np.random.randint(self.size_range[0], self.size_range[1] + 1)
            y = np.random.randint(0, max(1, h - hole_h))
            x = np.random.randint(0, max(1, w - hole_w))
            arr[y : y + hole_h, x : x + hole_w] = self.fill_value
        return Image.fromarray(arr)


print("Custom transforms definidos: RandomGamma, ShadesOfGray, CoarseDropout")

## 5. Transform Pipelines (TRAIN / VAL)

In [ ]:
TRANSFORM_TRAIN = transforms.Compose(
    [
        transforms.RandomCrop(TARGET_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(
            degrees=360,
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=0,
        ),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        RandomGamma(gamma_range=(0.7, 1.5), p=0.5),
        transforms.RandomApply([ShadesOfGray(power=6)], p=0.5),
        CoarseDropout(n_holes_range=(1, 3), size_range=(32, 96), p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

TRANSFORM_VAL = transforms.Compose(
    [
        transforms.CenterCrop(TARGET_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

print("TRANSFORM_TRAIN:", TRANSFORM_TRAIN)
print("\nTRANSFORM_VAL:", TRANSFORM_VAL)

## 6. CutMix / MixUp helpers

In [ ]:
def cutmix_data(
    x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """
    CutMix: recorta región rectangular de imagen A y la pega en imagen B.
    Returns: mixed_x, y_a, y_b, lam
    """
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    rand_index = torch.randperm(batch_size, device=x.device)
    y_a = y
    y_b = y[rand_index]
    _, _, H, W = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    x = x.clone()
    x[:, :, bby1:bby2, bbx1:bbx2] = x[rand_index, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - (bbx2 - bbx1) * (bby2 - bby1) / (W * H)
    return x, y_a, y_b, lam


def mixup_data(
    x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """
    MixUp: combina linealmente dos imágenes y sus etiquetas.
    Returns: mixed_x, y_a, y_b, lam
    """
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    rand_index = torch.randperm(batch_size, device=x.device)
    y_a = y
    y_b = y[rand_index]
    mixed_x = lam * x + (1.0 - lam) * x[rand_index]
    return mixed_x, y_a, y_b, lam


print("CutMix / MixUp helpers definidos")

## 7. ISICDataset (solo imágenes preprocesadas, autosuficiente)

In [ ]:
class ISICDataset(Dataset):
    """
    ISIC 2019 — Multiclase, 8 clases (ConvNeXt-Small).

    Modo "expert" para FASE 2: retorna (img, class_label_int, img_name).

    Usa ÚNICAMENTE imágenes preprocesadas (DullRazor + resize shorter_side=224).
    No soporta fallback a imágenes originales.
    """

    CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
    N_TRAIN_CLS = 8

    def _resolve_image_path(self, img_name: str) -> Path | None:
        """Resuelve la ruta de una imagen preprocesada.

        Returns:
            Path al archivo encontrado, o None si no se encuentra.
        """
        pp_suffix = f"_pp_{TARGET_SIZE}"

        # Candidatos en el directorio de preprocesadas
        candidates = [
            f"{img_name}{pp_suffix}.jpg",
            f"{img_name}.jpg",
        ]

        # Variantes sin _downsampled
        if img_name.endswith("_downsampled"):
            base = img_name[: -len("_downsampled")]
            candidates += [
                f"{base}{pp_suffix}.jpg",
                f"{base}.jpg",
            ]

        for filename in candidates:
            path = self.preprocessed_dir / filename
            if path.exists():
                return path

        # Último recurso: glob en preprocessed_dir
        base_id = (
            img_name.split("_downsampled")[0]
            if "_downsampled" in img_name
            else img_name
        )
        glob_patterns = [f"{img_name}*"]
        if base_id != img_name:
            glob_patterns.append(f"{base_id}*")

        for pattern in glob_patterns:
            matches = sorted(self.preprocessed_dir.glob(pattern))
            for match in matches:
                if match.is_file() and match.suffix.lower() in (
                    ".jpg", ".jpeg", ".png", ".bmp", ".tiff",
                ):
                    return match

        return None

    @staticmethod
    def get_weighted_sampler(split_df: pd.DataFrame, classes: list[str]):
        """WeightedRandomSampler para compensar desbalance NV>>DF/VASC en FASE 2."""
        if "label_idx" in split_df.columns:
            labels = split_df["label_idx"].values.astype(int)
        else:
            labels = split_df[classes].values.argmax(axis=1)
        counts = np.bincount(labels, minlength=len(classes)).astype(float)
        counts = np.maximum(counts, 1)
        w_cls = 1.0 / counts
        w_samp = w_cls[labels]
        return WeightedRandomSampler(
            weights=torch.tensor(w_samp, dtype=torch.float32),
            num_samples=len(w_samp),
            replacement=True,
        )

    def __init__(
        self,
        preprocessed_dir: Path,
        split_df: pd.DataFrame,
        split: str = "train",
    ):
        self.preprocessed_dir = Path(preprocessed_dir)
        self.expert_id = EXPERT_IDS["isic"]
        self.split = split
        self.df = split_df.reset_index(drop=True)
        self.class_weights = None

        # Seleccionar transform según split
        if split == "train":
            self.transform = TRANSFORM_TRAIN
        else:
            self.transform = TRANSFORM_VAL

        log.info(
            f"[ISIC] Dataset: {len(self.df):,} imgs | "
            f"split='{split}' | preprocessed_dir={self.preprocessed_dir}"
        )

        # Preparar etiquetas
        self.df = self.df.copy()
        if "label_idx" in self.df.columns:
            self.df["class_label"] = self.df["label_idx"].astype(int)
        else:
            self.df["class_label"] = self.df[self.CLASSES].values.argmax(axis=1)

        # Distribución de clases
        dist = self.df["class_label"].value_counts().sort_index()
        total = len(self.df)
        log.info("[ISIC] Distribución de clases:")
        for i, cls in enumerate(self.CLASSES):
            count = dist.get(i, 0)
            bar = "█" * max(1, int(30 * count / total))
            log.info(
                f"    {cls:<5}(idx {i}): {count:>6,} ({100 * count / total:.1f}%) {bar}"
            )

        # Class weights (inverse-frequency)
        counts_arr = np.array(
            [dist.get(i, 1) for i in range(self.N_TRAIN_CLS)], dtype=float
        )
        weights = total / (self.N_TRAIN_CLS * counts_arr)
        self.class_weights = torch.tensor(weights, dtype=torch.float32)
        log.info(
            f"[ISIC] class_weights[{self.N_TRAIN_CLS}]: "
            f"min={self.class_weights.min():.2f}({self.CLASSES[self.class_weights.argmin()]}) | "
            f"max={self.class_weights.max():.2f}({self.CLASSES[self.class_weights.argmax()]})"
        )

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        img_name = self.df.loc[idx, "image"]
        label = int(self.df.loc[idx, "class_label"])

        resolved_path = self._resolve_image_path(img_name)
        if resolved_path is None:
            raise FileNotFoundError(
                f"[ISIC] Imagen preprocesada no encontrada para '{img_name}' "
                f"en {self.preprocessed_dir}"
            )

        img = Image.open(resolved_path).convert("RGB")
        return self.transform(img), label, img_name


print(f"ISICDataset definido — {ISICDataset.N_TRAIN_CLS} clases: {ISICDataset.CLASSES}")

## 8. FocalLossMultiClass

In [ ]:
class FocalLossMultiClass(nn.Module):
    """
    Focal Loss multiclase para ISIC 2019 (Lin et al., ICCV 2017).

    Reduce el gradiente de ejemplos fáciles y concentra el aprendizaje
    en ejemplos difíciles de clases minoritarias (DF, VASC, AK, SCC).

    Args:
        gamma: concentración focal (default 2.0)
        weight: tensor[8] con pesos inverse-frequency por clase
        label_smoothing: reduce overconfidence (default 0.0)
    """

    def __init__(
        self,
        gamma: float = 2.0,
        weight: torch.Tensor | None = None,
        label_smoothing: float = 0.0,
    ):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.register_buffer("weight", weight)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction="none",
        )
        pt = torch.exp(-ce)
        focal_weight = (1.0 - pt) ** self.gamma
        return (focal_weight * ce).mean()


print("FocalLossMultiClass definido")

## 9. Modelo Expert2ConvNeXtSmall

In [ ]:
class Expert2ConvNeXtSmall(nn.Module):
    """
    ConvNeXt-Small adaptado para ISIC 2019 — 8 clases multiclase.

    Backbone timm convnext_small con pesos ImageNet pretrained.
    El head original se reemplaza por un clasificador personalizado
    con dos capas lineales, GELU, dropout y LayerNorm.

    Entrada:  [B, 3, 224, 224]
    Salida:   [B, 8] — logits crudos (antes de softmax)

    Head personalizado:
        LayerNorm(768) → Dropout(0.4) → Linear(768→256) → GELU()
        → Dropout(0.3) → Linear(256→num_classes)
    """

    def __init__(
        self,
        num_classes: int = 8,
        pretrained: bool = True,
    ) -> None:
        super().__init__()

        # Backbone ConvNeXt-Small (timm)
        # num_classes=0 + global_pool='avg' → forward retorna [B, 768]
        backbone = timm.create_model(
            "convnext_small",
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg",
        )
        self.model = backbone

        # Head personalizado sobre embeddings [B, 768]
        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(p=0.4),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

        # Inicialización de la capa final
        final_linear: nn.Linear = self.head[5]  # type: ignore[assignment]
        trunc_normal_(final_linear.weight, std=0.02)
        nn.init.zeros_(final_linear.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.model(x)   # [B, 768]
        logits = self.head(features)  # [B, num_classes]
        return logits

    # ── Métodos para optimizer diferencial ──────────────────────────

    def get_head_params(self):
        """Retorna parámetros del head (para lr más alto)."""
        return self.head.parameters()

    def get_backbone_params(self):
        """Retorna parámetros del backbone sin incluir el head."""
        return self.model.parameters()

    # ── Congelamiento / descongelamiento ────────────────────────────

    def freeze_backbone(self) -> None:
        """Congela todos los parámetros del backbone (self.model)."""
        for param in self.model.parameters():
            param.requires_grad = False

    def unfreeze_last_stages(self, n: int = 2) -> None:
        """
        Descongela los últimos n stages del backbone.
        ConvNeXt-Small tiene 4 stages (0-3). Con n=2 se descongelan stages 2 y 3.
        """
        stages = self.model.stages
        total_stages = len(stages)
        start = max(0, total_stages - n)
        for i in range(start, total_stages):
            for param in stages[i].parameters():
                param.requires_grad = True

    def unfreeze_all(self) -> None:
        """Descongela todos los parámetros del modelo (backbone + head)."""
        for param in self.parameters():
            param.requires_grad = True

    # ── Utilidades ──────────────────────────────────────────────────

    def count_parameters(self) -> int:
        """Número total de parámetros entrenables."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def count_all_parameters(self) -> int:
        """Número total de parámetros (entrenables + congelados)."""
        return sum(p.numel() for p in self.parameters())


print("Expert2ConvNeXtSmall definido")

## 10. DataLoaders con WeightedRandomSampler

In [ ]:
def build_dataloaders(
    preprocessed_dir=None,
    train_csv=None,
    val_csv=None,
    test_csv=None,
    batch_size=None,
    num_workers=EXPERT2_NUM_WORKERS,
):
    """
    Construye DataLoaders para train, val y test del Expert 2 (ISIC 2019).

    Usa únicamente imágenes preprocesadas. Si el directorio no existe, lanza error.

    Returns:
        (train_loader, val_loader, test_loader, class_weights)
    """
    preprocessed_dir = Path(preprocessed_dir) if preprocessed_dir else ISIC_DATA_DIR
    train_csv = Path(train_csv) if train_csv else ISIC_TRAIN_CSV
    val_csv = Path(val_csv) if val_csv else ISIC_VAL_CSV
    test_csv = Path(test_csv) if test_csv else ISIC_TEST_CSV
    batch_size = batch_size or EXPERT2_BATCH_SIZE

    # Verificar que los archivos/directorios existen
    for label, path in [
        ("preprocessed_dir", preprocessed_dir),
        ("train_csv", train_csv),
        ("val_csv", val_csv),
        ("test_csv", test_csv),
    ]:
        if not path.exists():
            raise FileNotFoundError(f"[DataLoader] {label} no encontrado: {path}")

    log.info(f"[DataLoader] Preprocessed images: {preprocessed_dir}")
    log.info(f"[DataLoader] Batch: {batch_size} | Workers: {num_workers}")

    # Cargar DataFrames
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)

    log.info(
        f"[DataLoader] Splits — "
        f"train: {len(train_df):,} | val: {len(val_df):,} | test: {len(test_df):,}"
    )

    # Crear datasets
    train_ds = ISICDataset(
        preprocessed_dir=preprocessed_dir,
        split_df=train_df,
        split="train",
    )
    val_ds = ISICDataset(
        preprocessed_dir=preprocessed_dir,
        split_df=val_df,
        split="val",
    )
    test_ds = ISICDataset(
        preprocessed_dir=preprocessed_dir,
        split_df=test_df,
        split="test",
    )

    # Obtener class_weights
    if hasattr(train_ds, "class_weights") and train_ds.class_weights is not None:
        class_weights = train_ds.class_weights
        log.info(
            f"[DataLoader] class_weights: "
            f"shape={class_weights.shape} | "
            f"min={class_weights.min():.3f} | max={class_weights.max():.3f}"
        )
    else:
        # Fallback
        labels = [train_ds[i][1] for i in range(len(train_ds))]
        n_classes = ISICDataset.N_TRAIN_CLS
        counts = np.bincount(labels, minlength=n_classes)
        class_weights = torch.tensor(
            len(train_ds) / (n_classes * np.maximum(counts, 1)),
            dtype=torch.float32,
        )

    # WeightedRandomSampler para train
    sampler = ISICDataset.get_weighted_sampler(train_df, ISICDataset.CLASSES)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=sampler,
        drop_last=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
    )

    # Resumen
    print("=" * 70)
    print("  RESUMEN — DataLoaders Expert 2 (ISIC 2019)")
    print("=" * 70)
    for name, ds, loader in [
        ("train", train_ds, train_loader),
        ("val", val_ds, val_loader),
        ("test", test_ds, test_loader),
    ]:
        print(f"  {name:5s}: {len(ds):>6,} muestras | batches={len(loader):>4,}")
    print("=" * 70)

    return train_loader, val_loader, test_loader, class_weights


print("build_dataloaders() definido")

## 11. Funciones de entrenamiento y validación

In [ ]:
def set_seed(seed: int = SEED) -> None:
    """Fija todas las semillas para reproducibilidad."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    log.info(f"[Seed] Semillas fijadas a {seed}")


def log_vram(tag: str = "") -> None:
    """Imprime uso actual de VRAM si hay GPU disponible."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        log.info(
            f"[VRAM{' ' + tag if tag else ''}] "
            f"Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB"
        )


class EarlyStoppingF1:
    """
    Early stopping por val_f1_macro (maximizar) con patience configurable.
    """

    def __init__(self, patience: int, min_delta: float = 0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = -float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_f1_macro: float) -> bool:
        if val_f1_macro > self.best_score + self.min_delta:
            self.best_score = val_f1_macro
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                return True
            return False


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    accumulation_steps,
    use_fp16,
    cutmix_prob: float = 0.3,
    mixup_prob: float = 0.2,
) -> float:
    """
    Una época de entrenamiento con FP16, gradient accumulation,
    gradient clipping y CutMix/MixUp.

    Returns:
        Loss promedio de la época.
    """
    model.train()
    total_loss = 0.0
    n_batches = 0

    optimizer.zero_grad()

    for batch_idx, (imgs, labels, _stems) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.long().to(device, non_blocking=True)

        # Selección de augmentación de batch (mutuamente excluyentes)
        r = np.random.random()
        use_cutmix = r < cutmix_prob
        use_mixup = (not use_cutmix) and (r < cutmix_prob + mixup_prob)

        # Forward con autocast FP16
        with torch.amp.autocast(device_type=device.type, enabled=use_fp16):
            if use_cutmix:
                imgs, y_a, y_b, lam = cutmix_data(imgs, labels)
                logits = model(imgs)
                loss = lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)
            elif use_mixup:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels)
                logits = model(imgs)
                loss = lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)
            else:
                logits = model(imgs)
                loss = criterion(logits, labels)
            loss = loss / accumulation_steps

        # Backward
        scaler.scale(loss).backward()

        # Optimizer step cada accumulation_steps batches
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accumulation_steps
        n_batches += 1

    # Flush de gradientes residuales
    if n_batches % accumulation_steps != 0:
        scaler.unscale_(optimizer)
        clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss


@torch.no_grad()
def validate(model, loader, criterion, device, use_fp16) -> dict:
    """
    Validación con métricas: val_loss, val_acc, val_f1_macro, val_bmca, val_auc.
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0

    all_logits = []
    all_labels = []

    for batch_idx, (imgs, labels, _stems) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        labels_dev = labels.long().to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_fp16):
            logits = model(imgs)
            loss = criterion(logits, labels_dev)

        total_loss += loss.item()
        n_batches += 1

        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    avg_loss = total_loss / max(n_batches, 1)

    # Métricas
    all_logits_t = torch.cat(all_logits, dim=0)
    all_probs = torch.softmax(all_logits_t, dim=1).numpy()
    all_preds = all_probs.argmax(axis=1)
    all_labels_np = torch.cat(all_labels, dim=0).numpy()

    acc = float(np.mean(all_preds == all_labels_np))
    f1_macro = float(
        f1_score(all_labels_np, all_preds, average="macro", zero_division=0)
    )
    bmca = float(balanced_accuracy_score(all_labels_np, all_preds))

    try:
        auc = float(
            roc_auc_score(
                all_labels_np,
                all_probs[:, :EXPERT2_NUM_CLASSES],
                multi_class="ovr",
                average="macro",
                labels=list(range(EXPERT2_NUM_CLASSES)),
            )
        )
    except ValueError:
        auc = 0.0
        log.warning("[Val] AUC-ROC no computable → AUC=0.0")

    return {
        "val_loss": avg_loss,
        "val_acc": acc,
        "val_f1_macro": f1_macro,
        "val_bmca": bmca,
        "val_auc": auc,
    }


print("train_one_epoch(), validate(), EarlyStoppingF1, set_seed(), log_vram() definidos")

## 12. Training Loop Trifásico

In [ ]:
def run_phase(
    phase_num,
    phase_name,
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    device,
    use_fp16,
    num_epochs,
    global_epoch_offset,
    best_f1_macro,
    training_log,
    early_stopping=None,
    cutmix_prob=0.3,
    mixup_prob=0.2,
):
    """
    Ejecuta una fase completa de entrenamiento.

    Returns:
        Mejor val_f1_macro alcanzado (actualizado si mejoró).
    """
    log.info(f"\n{'=' * 70}")
    log.info(f"  FASE {phase_num}: {phase_name}")
    log.info(
        f"  Épocas: {num_epochs} (global {global_epoch_offset + 1}"
        f"-{global_epoch_offset + num_epochs})"
    )
    log.info(f"  Params entrenables: {model.count_parameters():,}")
    for i, pg in enumerate(optimizer.param_groups):
        log.info(
            f"  Param group {i}: lr={pg['lr']:.2e}, wd={pg.get('weight_decay', 0):.1e}"
        )
    log.info(f"{'=' * 70}\n")

    for epoch_local in range(num_epochs):
        epoch_global = global_epoch_offset + epoch_local + 1
        epoch_start = time.time()

        # Train
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            accumulation_steps=EXPERT2_ACCUMULATION_STEPS,
            use_fp16=use_fp16,
            cutmix_prob=cutmix_prob,
            mixup_prob=mixup_prob,
        )

        # Validation
        val_results = validate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            use_fp16=use_fp16,
        )

        # Scheduler step
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

        # Extraer métricas
        epoch_time = time.time() - epoch_start
        val_loss = val_results["val_loss"]
        val_acc = val_results["val_acc"]
        val_f1_macro = val_results["val_f1_macro"]
        val_bmca = val_results["val_bmca"]
        val_auc = val_results["val_auc"]

        is_best = val_f1_macro > best_f1_macro

        log.info(
            f"[Epoch {epoch_global:3d}/{EXPERT2_TOTAL_EPOCHS} | F{phase_num}] "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | val_f1_macro={val_f1_macro:.4f} | "
            f"val_bmca={val_bmca:.4f} | val_auc={val_auc:.4f} | "
            f"lr={current_lr:.2e} | time={epoch_time:.1f}s"
            f"{'  ★ BEST' if is_best else ''}"
        )
        log_vram(f"epoch-{epoch_global}")

        # Log de métricas
        epoch_log = {
            "epoch": epoch_global,
            "phase": phase_num,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1_macro": val_f1_macro,
            "val_bmca": val_bmca,
            "val_auc": val_auc,
            "lr": current_lr,
            "epoch_time_s": round(epoch_time, 1),
            "is_best": is_best,
        }
        training_log.append(epoch_log)

        # Guardar mejor checkpoint (por val_f1_macro)
        if is_best:
            best_f1_macro = val_f1_macro
            checkpoint = {
                "epoch": epoch_global,
                "phase": phase_num,
                "model_state_dict": model.state_dict(),
                "val_f1_macro": val_f1_macro,
                "val_bmca": val_bmca,
                "val_auc": val_auc,
                "val_loss": val_loss,
            }
            torch.save(checkpoint, CHECKPOINT_PATH)
            log.info(f"  → Checkpoint guardado: {CHECKPOINT_PATH}")

        # Guardar training log
        with open(TRAINING_LOG_PATH, "w") as f:
            json.dump(training_log, f, indent=2)

        # Early stopping
        if early_stopping is not None:
            if early_stopping.step(val_f1_macro):
                log.info(
                    f"\n[EarlyStopping] Detenido en época {epoch_global}. "
                    f"val_f1_macro no mejoró en {early_stopping.patience} épocas. "
                    f"Mejor val_f1_macro: {best_f1_macro:.4f}"
                )
                break

    # Resumen de fase
    phase_logs = [e for e in training_log if e["phase"] == phase_num]
    if phase_logs:
        best_epoch_log = max(phase_logs, key=lambda x: x["val_f1_macro"])
        log.info(
            f"\n[Fase {phase_num} resumen] Mejor época: {best_epoch_log['epoch']} | "
            f"val_f1_macro={best_epoch_log['val_f1_macro']:.4f} | "
            f"val_bmca={best_epoch_log['val_bmca']:.4f} | "
            f"val_auc={best_epoch_log['val_auc']:.4f}"
        )

    return best_f1_macro


print("run_phase() definido")

## 13. Ejecutar entrenamiento completo

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Semilla + dispositivo
# ══════════════════════════════════════════════════════════════════════
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"[Expert2] Dispositivo: {device}")
if device.type == "cpu":
    log.warning(
        "[Expert2] ⚠ Entrenando en CPU — será muy lento. "
        "Se recomienda GPU con >= 12 GB VRAM."
    )

use_fp16 = device.type == "cuda"
if not use_fp16:
    log.info("[Expert2] FP16 desactivado (no hay GPU). Usando FP32 en CPU.")

# ══════════════════════════════════════════════════════════════════════
#  Modelo
# ══════════════════════════════════════════════════════════════════════
model = Expert2ConvNeXtSmall(
    num_classes=EXPERT2_NUM_CLASSES,
    pretrained=True,
).to(device)

total_params = model.count_all_parameters()
log.info(f"[Expert2] Modelo ConvNeXt-Small creado: {total_params:,} parámetros totales")
log_vram("post-model")

# ══════════════════════════════════════════════════════════════════════
#  DataLoaders
# ══════════════════════════════════════════════════════════════════════
train_loader, val_loader, _test_loader, class_weights = build_dataloaders(
    batch_size=EXPERT2_BATCH_SIZE,
    num_workers=EXPERT2_NUM_WORKERS,
)
class_weights = class_weights.to(device)

# ══════════════════════════════════════════════════════════════════════
#  Loss (compartida entre las 3 fases)
# ══════════════════════════════════════════════════════════════════════
criterion = FocalLossMultiClass(
    gamma=2.0,
    weight=class_weights,
    label_smoothing=EXPERT2_LABEL_SMOOTHING,
)
log.info(
    f"[Expert2] Loss: FocalLossMultiClass("
    f"gamma=2.0, weight=class_weights[{class_weights.shape[0]}], "
    f"label_smoothing={EXPERT2_LABEL_SMOOTHING})"
)

# GradScaler para FP16 (compartido entre fases)
scaler = GradScaler(device=device.type, enabled=use_fp16)

# Directorio de checkpoints
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Estado global
best_f1_macro = -float("inf")
training_log: list[dict] = []

log.info(f"\n{'=' * 70}")
log.info(f"  INICIO DE ENTRENAMIENTO — Expert 2 (ConvNeXt-Small / ISIC 2019)")
log.info(
    f"  Total épocas: {EXPERT2_TOTAL_EPOCHS} | Batch efectivo: "
    f"{EXPERT2_BATCH_SIZE}×{EXPERT2_ACCUMULATION_STEPS}="
    f"{EXPERT2_BATCH_SIZE * EXPERT2_ACCUMULATION_STEPS}"
)
log.info(
    f"  FP16: {use_fp16} | Accumulation: {EXPERT2_ACCUMULATION_STEPS} | "
    f"Grad clip: {GRAD_CLIP_NORM}"
)
log.info(
    f"  3 fases: head-only({EXPERT2_PHASE1_EPOCHS}) → "
    f"partial({EXPERT2_PHASE2_EPOCHS}) → full({EXPERT2_PHASE3_EPOCHS})"
)
log.info(f"{'=' * 70}\n")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FASE 1: Solo head, backbone congelado (épocas 1-5)
# ══════════════════════════════════════════════════════════════════════
model.freeze_backbone()
log.info(
    f"[Fase 1] freeze_backbone() → "
    f"{model.count_parameters():,} params entrenables (solo head)"
)

optimizer_p1 = torch.optim.AdamW(
    model.get_head_params(),
    lr=EXPERT2_PHASE1_LR,
    weight_decay=EXPERT2_PHASE1_WD,
)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1,
    T_max=EXPERT2_PHASE1_EPOCHS,
    eta_min=EXPERT2_PHASE1_ETA_MIN,
)

best_f1_macro = run_phase(
    phase_num=1,
    phase_name="Head-only (backbone congelado)",
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_p1,
    scheduler=scheduler_p1,
    scaler=scaler,
    device=device,
    use_fp16=use_fp16,
    num_epochs=EXPERT2_PHASE1_EPOCHS,
    global_epoch_offset=0,
    best_f1_macro=best_f1_macro,
    training_log=training_log,
    early_stopping=None,
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FASE 2: Fine-tuning diferencial (últimos 2 stages + head) (épocas 6-20)
# ══════════════════════════════════════════════════════════════════════
model.unfreeze_last_stages(n=2)
log.info(
    f"[Fase 2] unfreeze_last_stages(n=2) → "
    f"{model.count_parameters():,} params entrenables"
)

optimizer_p2 = torch.optim.AdamW(
    [
        {"params": model.get_head_params(), "lr": EXPERT2_PHASE2_HEAD_LR},
        {"params": model.get_backbone_params(), "lr": EXPERT2_PHASE2_BACKBONE_LR},
    ],
    weight_decay=EXPERT2_PHASE2_WD,
)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p2,
    T_0=EXPERT2_PHASE2_T0,
    T_mult=EXPERT2_PHASE2_T_MULT,
    eta_min=EXPERT2_PHASE2_ETA_MIN,
)

best_f1_macro = run_phase(
    phase_num=2,
    phase_name="Fine-tuning diferencial (últimos 2 stages + head)",
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_p2,
    scheduler=scheduler_p2,
    scaler=scaler,
    device=device,
    use_fp16=use_fp16,
    num_epochs=EXPERT2_PHASE2_EPOCHS,
    global_epoch_offset=EXPERT2_PHASE1_EPOCHS,
    best_f1_macro=best_f1_macro,
    training_log=training_log,
    early_stopping=None,
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FASE 3: Full fine-tuning + early stopping (épocas 21-40)
# ══════════════════════════════════════════════════════════════════════
model.unfreeze_all()
log.info(
    f"[Fase 3] unfreeze_all() → "
    f"{model.count_parameters():,} params entrenables (todo descongelado)"
)

optimizer_p3 = torch.optim.AdamW(
    [
        {"params": model.get_head_params(), "lr": EXPERT2_PHASE3_HEAD_LR},
        {"params": model.get_backbone_params(), "lr": EXPERT2_PHASE3_BACKBONE_LR},
    ],
    weight_decay=EXPERT2_PHASE3_WD,
)
scheduler_p3 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p3,
    T_0=EXPERT2_PHASE3_T0,
    T_mult=EXPERT2_PHASE3_T_MULT,
    eta_min=EXPERT2_PHASE3_ETA_MIN,
)

early_stopping = EarlyStoppingF1(
    patience=EXPERT2_EARLY_STOPPING_PATIENCE,
    min_delta=MIN_DELTA,
)
log.info(
    f"[Fase 3] EarlyStopping: monitor=val_f1_macro, "
    f"patience={EXPERT2_EARLY_STOPPING_PATIENCE}, min_delta={MIN_DELTA}"
)

best_f1_macro = run_phase(
    phase_num=3,
    phase_name="Full fine-tuning + early stopping",
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_p3,
    scheduler=scheduler_p3,
    scaler=scaler,
    device=device,
    use_fp16=use_fp16,
    num_epochs=EXPERT2_PHASE3_EPOCHS,
    global_epoch_offset=EXPERT2_PHASE1_EPOCHS + EXPERT2_PHASE2_EPOCHS,
    best_f1_macro=best_f1_macro,
    training_log=training_log,
    early_stopping=early_stopping,
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Resumen final
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 70}")
print(f"  ENTRENAMIENTO FINALIZADO — Expert 2 (ConvNeXt-Small / ISIC 2019)")
print(f"  Mejor val_f1_macro: {best_f1_macro:.4f}")
if training_log:
    best_epoch = max(training_log, key=lambda x: x["val_f1_macro"])
    print(
        f"  Mejor época: {best_epoch['epoch']} (fase {best_epoch['phase']}) | "
        f"F1-macro: {best_epoch['val_f1_macro']:.4f} | "
        f"BMCA: {best_epoch['val_bmca']:.4f} | "
        f"AUC: {best_epoch['val_auc']:.4f}"
    )
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Training log: {TRAINING_LOG_PATH}")
print(f"{'=' * 70}")